## Initialization

In [1]:
#Imports
import math
import torch
import torchaudio
from torch import nn
from torch.utils.data import DataLoader
#from torchvision import transforms

#Custom modules
import cAudiotools
import cLogger
import cTransforms
import cModelManager
#import cNNModules

MODEL_NAME = "WavLM_base_2_VAD_baseline"
MODELS_DIR = "output/models"
model_description = "WavLM for VAD regression with the base plus version using only output embeddings."

#Define output paths
log = cLogger.Log("output/logs", prefix=MODEL_NAME)
modelMngr = cModelManager.ModelManager(f"{MODELS_DIR}/{MODEL_NAME}")
log.log_property("model_name", MODEL_NAME)
log.log_property("model_description", model_description, show=False)
log.log_property("model_dir", modelMngr.model_directory, show=False)

#Torch properties and device
log.log_property("torch_version", torch.__version__)
log.log_property("torchaudio_version", torchaudio.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    log.log_property("device", "cuda")
    log.log_property("GPU_count", torch.cuda.device_count())
    log.log_property("GPU_device", torch.cuda.get_device_name(0))
else:
    log.log_property("device", "cpu")
    

Directory created at: output/models/WavLM_base_2_VAD_baseline\run_2026_01_28-183133
model_name: WavLM_base_2_VAD_baseline
torch_version: 2.7.1+cu128
torchaudio_version: 2.7.1+cu128
device: cuda
GPU_count: 1
GPU_device: NVIDIA GeForce GTX 1050


## Define data parameters

In [2]:
#Parameters:
loader_params = {
    "dataset_train": r"data/labels_msp_consensus_undersampled.csv",
    "dataset_train_dir": r"C:/Datasets/_compiled/msp-podcast-2/Train",
    "dataset_dev": r"C:/Datasets/_compiled/msp-podcast-2/labels/custom/labels_development_VAD.csv",
    "dataset_dev_dir": r"C:/Datasets/_compiled/msp-podcast-2/Development",
    "shuffle": True,
    "collate_function": cAudiotools.Collate.waveform_dynamic_wMasks,
    "data_transform": None,
    "target_transform": cTransforms.NormalizeMinus(1, 7)
}

training_params = {
    "epochs": 50,
    "batch_size": 16,
    "checkpoint_interval": 5,
    "checkpoint_before_training": True,
    "criterion_for_best": "val_avg_loss",
}

optimizer_params = {    
    "learning_rate": 1e-3,
    "adam_betas": (0.9, 0.999),
    "adam_epsilon": 1e-8,
}

#scheduler_params = {
#    "use_scheduler": True,
#    "scheduler": torch.optim.lr_scheduler.ReduceLROnPlateau(),
#}

audio_params = {
    "sample_rate": 16000
}

#spectrogram_params = {
#    "n_fft": 512,
#    "hop_length": 160,
#    "win_length": 400,
#    "window_fn": torch.hamming_window,
#    "normalized": True,
#    "power": 2
#}
#
#mel_params = {
#    "n_mels": 64,
#    "pad_mode": "constant",
#    "mel_scale": "htk",
#    "n_mfcc": 20
#}


## Define training parameters

In [3]:

#spectogram_transform = torchaudio.transforms.Spectrogram(
#    n_fft=spectrogram_params["n_fft"],
#    hop_length=spectrogram_params["hop_length"],
#    win_length=spectrogram_params["win_length"],
#    window_fn=spectrogram_params["window_fn"],
#    normalized=spectrogram_params["normalized"],
#    power=spectrogram_params["power"]
#    )
#
#melspectogram_transform = torchaudio.transforms.MelSpectrogram(
#    sample_rate=audio_params["sample_rate"],
#    n_fft=spectrogram_params["n_fft"],
#    win_length=spectrogram_params["win_length"],
#    hop_length=spectrogram_params["hop_length"],
#    window_fn=spectrogram_params["window_fn"],
#    power=spectrogram_params["power"],
#    n_mels=mel_params["n_mels"],
#    normalized=spectrogram_params["normalized"],
#    pad_mode=mel_params["pad_mode"],
#    mel_scale=mel_params["mel_scale"]
#)
#
#mfcc_transform = torchaudio.transforms.MFCC(
#    sample_rate=audio_params["sample_rate"],
#    n_mfcc=mel_params["n_mfcc"],
#    melkwargs={
#        "n_fft": spectrogram_params["n_fft"],
#        "win_length": spectrogram_params["win_length"],
#        "hop_length": spectrogram_params["hop_length"],
#        "window_fn": spectrogram_params["window_fn"],
#        "power": spectrogram_params["power"],
#        "n_mels": mel_params["n_mels"],
#        "normalized": spectrogram_params["normalized"],
#        "pad_mode": mel_params["pad_mode"],
#        "mel_scale": mel_params["mel_scale"]
#    }
#)
#loader_params["data_transform"] = None

log.log_properties("Loader", loader_params, show=False)
log.log_properties("Training", training_params, show=False)
log.log_properties("Audio", audio_params, show=False)
#log.log_properties("Spectrogram", spectrogram_params)
#log.log_properties("Mel Spectrogram", mel_params)

#Train set
dataset_train = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_train"], 
    loader_params["dataset_train_dir"],
    transform=loader_params["data_transform"],
    target_transform=loader_params["target_transform"]
    )
dataset_train_loader = DataLoader(
    dataset_train,
    batch_size=training_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"],
    pin_memory=True 
    )

#Development (validation) set
dataset_dev = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_dev"],
    loader_params["dataset_dev_dir"],
    transform=loader_params["data_transform"],
    target_transform=loader_params["target_transform"]
    )
dataset_dev_loader = DataLoader(
    dataset_dev,
    batch_size=training_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"],
    pin_memory=True 
    )


## Data check

In [4]:
log.log_message(f"Train samples: {len(dataset_train)}")
log.log_message(f"Dev samples: {len(dataset_dev)}")
log.log_message(f"Batch size: {training_params['batch_size']}")

sample_batch = next(iter(dataset_train_loader))
inputs, masks, targets = sample_batch

log.log_message("Sample batch:")
log.log_message(f"Inputs Shape: {inputs.shape}")
log.log_message(f"Targets Shape: {targets.shape}")
log.log_message(f"Masks Shape: {masks.shape}")
log.log_message(f"Input range: Min={inputs.min():.2f}, Max={inputs.max():.2f}")
log.log_message(f"Output range: Min={targets.min():.2f}, Max={targets.max():.2f}")

Train samples: 5000
Dev samples: 34399
Batch size: 16
Sample batch:
Inputs Shape: torch.Size([16, 165888])
Targets Shape: torch.Size([16, 3])
Masks Shape: torch.Size([16, 165888])
Input range: Min=-0.94, Max=0.94
Output range: Min=-0.53, Max=0.53


## Define Architecture

In [6]:
from transformers import WavLMModel

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
        self.hidden_size = self.wavlm.config.hidden_size
        #Freeze weights
        self.wavlm.freeze_feature_encoder()
        for param in self.wavlm.parameters():
            param.requires_grad = False

        self.regression_head = nn.Sequential(
            nn.Linear(self.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
            nn.Tanh()
        )
    
    def frame_pooling(self, features, attention_masks):        
        if attention_masks is not None:
            feat_len = features.size(1)
            
            # Add dimensions for interpolation: (Batch, 1, Time)
            m = attention_masks.unsqueeze(1).float()
            
            # Downsample mask using Nearest Neighbor
            m = nn.functional.interpolate(m, size=feat_len, mode='nearest')
            
            # Transpose to match features: (Batch, Time, 1)
            m = m.transpose(1, 2)

            features = features * m
            pooled = features.sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = features.mean(dim=1)
            
        return pooled

    def forward(self, input, attention_masks):
        self.wavlm.eval()
        with torch.no_grad():
            ssl_output = self.wavlm(input, attention_mask=attention_masks)
            features = ssl_output.last_hidden_state  # (batch_size, seq_len, hidden_size)
        
        frame_pooled_embedding = self.frame_pooling(features, attention_masks)
        logits = self.regression_head(frame_pooled_embedding)
        return logits


model = NeuralNetwork().to(device)
log.log_property("model_structure", str(model))

loss_fn = nn.MSELoss()
log.log_property("loss_function", str(loss_fn))

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=optimizer_params["learning_rate"], 
    betas=optimizer_params["adam_betas"],
    eps=optimizer_params["adam_epsilon"]
    )

log.log_property("optimizer", str(optimizer))

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model_structure: NeuralNetwork(
  (wavlm): WavLMModel(
    (feature_extractor): WavLMFeatureEncoder(
      (conv_layers): ModuleList(
        (0): WavLMGroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x WavLMNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x WavLMNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): WavLMFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): Wav

## Training

In [ ]:
#Loop definitions
def train_loop(dataloader, model, loss_fn, optimizer, metrics_dict=None):
    model.train()
    size = len(dataloader.dataset)
    epoch_loss = 0
    for batch, (inputs, masks, targets) in enumerate(dataloader):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        pred = model(inputs, masks)
        loss = loss_fn(pred, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * inputs.size(0)
        batch_loss = loss.item()

        if batch % 100 == 0:
            current = batch * len(inputs)
            print(f"[{current:>6d}/{size:>6d}] batch loss: {batch_loss:>7f}")
    
    epoch_loss /= size
    if metrics_dict:
        metrics_dict["train_avg_loss"] = epoch_loss

def validation_loop(dataloader, model, loss_fn, metrics_dict=None):
    model.eval()
    size = len(dataloader.dataset)
    test_loss = 0

    with torch.no_grad():
        for inputs, masks, targets  in dataloader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            pred = model(inputs, masks)
            loss = loss_fn(pred, targets)
            test_loss += loss.item() * inputs.size(0)

    test_loss /= size
    if metrics_dict:
        metrics_dict["val_avg_loss"] = test_loss


#Set Model Manager
modelMngr.set_model(model, optimizer, training_params["criterion_for_best"])

#Training
epoch_metrics = {'train_avg_loss': math.inf, 'val_avg_loss': math.inf}
remaining_for_checkpoint = training_params["checkpoint_interval"]

if training_params["checkpoint_before_training"]:
        modelMngr.checkpoint(0, epoch_metrics)

log.track_time(True, message="Starting training.")
total_epochs = training_params["epochs"]

for epoch in range(total_epochs):
    log.log_message(f"Epoch {epoch + 1} of {total_epochs}...")
    
    train_loop(dataset_train_loader, model, loss_fn, optimizer, metrics_dict=epoch_metrics)
    validation_loop(dataset_dev_loader, model, loss_fn, metrics_dict=epoch_metrics)

    log.log_epoch(epoch + 1, epoch_metrics)
    log.save()
    
    #Checkpointing
    remaining_for_checkpoint -= 1
    if remaining_for_checkpoint == 0:
        modelMngr.checkpoint(epoch + 1, epoch_metrics)
        remaining_for_checkpoint = training_params["checkpoint_interval"]
    
    #Save best model
    modelMngr.check_best(epoch + 1, epoch_metrics)

log.log_elapsed_time(message="Training completed")
log.track_time(False, show=False)
log.log_properties("Last_epoch", epoch_metrics)
log.log_properties("Best_model", modelMngr.best_model_metrics)
log.save()
#log.save_txt()

modelMngr.save_for_inference()
modelMngr.save_best(save_for_inference=True)


## Evaluation

In [ ]:
from torchmetrics.regression import ConcordanceCorrCoef
from sklearn.metrics import mean_squared_error, mean_absolute_error

evaluation_params = {
    "dataset_test": r"C:/Datasets/_compiled/msp-podcast-2/labels_test1_VAD.csv",
    "dataset_test_dir": r"C:/Datasets/_compiled/msp-podcast-2/Test1",
    "shuffle": False,
    "collate_function": cAudiotools.Collate.waveform_dynamic_wLengths,
    "data_transform": None,
    "target_transform": cTransforms.NormalizeMinus(1, 7),
    "batch_size": 32
}

log.log_properties("Evaluation", evaluation_params, show=False)

#Test set
dataset_test = cAudiotools.AudioDatasetVAD(
    evaluation_params["dataset_test"],
    evaluation_params["dataset_test_dir"],
    transform=evaluation_params["data_transform"],
    target_transform=evaluation_params["target_transform"]
    )
dataset_test_loader = DataLoader(
    dataset_test,
    batch_size=evaluation_params["batch_size"],
    shuffle=evaluation_params["shuffle"],
    collate_fn=evaluation_params["collate_function"]
)

log.log_message(f"Test samples: {len(dataset_test)}")
sample_batch = next(iter(dataset_test_loader))
inputs, masks, targets = sample_batch

log.log_message("Sample batch:")
log.log_message(f"Inputs Shape: {inputs.shape}")
log.log_message(f"Targets Shape: {targets.shape}")
log.log_message(f"Lengths Shape: {masks.shape}")
log.log_message(f"Input range: Min={inputs.min():.2f}, Max={inputs.max():.2f}")
log.log_message(f"Output range: Min={targets.min():.2f}, Max={targets.max():.2f}")

#Test loop
def test_loop(dataloader, model, device):
    total_predictions = []
    total_targets = []
    model.eval()

    with torch.no_grad():
        print("Starting evaluation...")
        size = len(dataloader.dataset)
        for batch, (inputs, lengths, targets) in enumerate(dataloader):
            inputs = inputs.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)

            pred = model(inputs, lengths)
            total_predictions.append(pred.cpu())
            total_targets.append(targets)
            if batch % 10 == 0:
                current = batch * len(inputs)
                print(f"[{current:>6d}/{size:>6d}] processed")
        final_predictions = torch.cat(total_predictions, dim=0)
        final_targets = torch.cat(total_targets, dim=0)
        return final_predictions, final_targets

predictions, targets = test_loop(dataset_test_loader, model, device)

log.log_message(f"Predictions Shape: {predictions.shape}")
log.log_message(f"Targets Shape: {targets.shape}")

concordance = ConcordanceCorrCoef(num_outputs=3)

results = {
    "Concordance_Correlation_Coefficient": concordance(predictions, targets).tolist(),
    "Mean_Squared_Error": mean_squared_error(targets.numpy(), predictions.numpy()),
    "Mean_Absolute_Error": mean_absolute_error(targets.numpy(), predictions.numpy())
}

log.log_properties("Test_results", results)
log.save()
log.save_txt()